In [6]:
# Cell 0: synchronize the repository environment before running the workbench.
import subprocess
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Find the workspace root without confusing member pyproject files for it."""
    for candidate in (start, *start.parents):
        if (
            (candidate / "uv.lock").exists()
            and (candidate / "implementations").is_dir()
            and (candidate / "aieng-forecasting").is_dir()
        ):
            return candidate
    raise FileNotFoundError("Could not find the agentic-forecasting repository root.")


SETUP_ROOT = find_repo_root(Path.cwd().resolve())
print("Running: uv sync --all-extras --dev --all-packages")
setup_result = subprocess.run(
    ["uv", "sync", "--all-extras", "--dev", "--all-packages"],
    cwd=SETUP_ROOT,
    check=False,
    text=True,
    capture_output=True,
)
print(setup_result.stdout)
if setup_result.returncode != 0:
    print(setup_result.stderr)
    raise RuntimeError(
        "uv sync failed. Install uv or restart the notebook with the repository environment selected."
    )
print("Environment synchronized successfully.")
print("If packages changed, restart the notebook kernel before continuing.")


Running: uv sync --all-extras --dev --all-packages

Environment synchronized successfully.
If packages changed, restart the notebook kernel before continuing.


# Manufacturing Stress Forecasting Workbench

This notebook is the main interactive entry point for the manufacturing-stress experiment. It creates or refreshes the FRED data cache, runs the deterministic baselines, and optionally runs the token-limited LLMP backtest.

All predictors use the same backtest specification and shared Brier-score implementation. Complete LLMP results are cached by backtest-specification fingerprint so repeated analysis does not make new model calls. Historical agent prompts use relative month offsets instead of calendar dates, but this remains a retrospective pseudo-backtest: model-training knowledge leakage cannot be eliminated completely.


In [7]:
# Main controls: change these values, then run the notebook from top to bottom.
REFRESH_FRED_DATA = True
BACKTEST_STRIDE = 3
RUN_LLMP_BACKTEST = False
FORCE_REFRESH_LLMP_CACHE = False
RUN_CURRENT_FORECAST = False

# BACKTEST_STRIDE = 1 evaluates every month; 3 evaluates every third month.
# False: one smoke run for historical frequency, logistic regression, and XGBoost.
# True: the same smoke run also includes the cached, token-limited LLMP.
# RUN_CURRENT_FORECAST controls the separate latest-data forecast section.
print({
    "refresh_fred_data": REFRESH_FRED_DATA,
    "backtest_stride": BACKTEST_STRIDE,
    "run_llmp_backtest": RUN_LLMP_BACKTEST,
    "force_refresh_llmp_cache": FORCE_REFRESH_LLMP_CACHE,
    "run_current_forecast": RUN_CURRENT_FORECAST,
})

{'refresh_fred_data': True, 'backtest_stride': 3, 'run_llmp_backtest': False, 'force_refresh_llmp_cache': False, 'run_current_forecast': False}


## Imports and project paths

Run this notebook with the repository environment selected as the Jupyter kernel. The `.env` file is loaded for the FRED key and, when needed, the LLM proxy credentials.

In [8]:
from pathlib import Path

import pandas as pd
import yaml
from aieng.forecasting.evaluation import BacktestSpec, backtest
from aieng.forecasting.methods import HistoricalFrequencyPredictor
from dotenv import load_dotenv
from manufacturing_stress_forecasting.analyst_agent import build_manufacturing_stress_agent_predictor
from manufacturing_stress_forecasting.data import IPMAN_SERIES_ID, build_manufacturing_stress_service
from manufacturing_stress_forecasting.predictors import (
    ManufacturingStressLogisticPredictor,
    ManufacturingStressXGBoostPredictor,
)
from manufacturing_stress_forecasting.run_agent_backtest import (
    run_or_load_backtest,
    validate_comparable_results,
)


REPO_ROOT = SETUP_ROOT
load_dotenv(REPO_ROOT / ".env", override=False)
SPEC_PATH = REPO_ROOT / "implementations" / "manufacturing_stress_forecasting" / "specs" / "manufacturing_stress_smoke.yaml"
STORE_DIR = REPO_ROOT / "data" / "predictions"
print(f"Repository: {REPO_ROOT}")


Repository: /home/coder/BMO-C2-agentic-forecasting


## Create or refresh the data

The service loads the requested FRED fields and the `SPY` and `XLI` adjusted-close histories from the local FRED and Yahoo Finance caches. Set `REFRESH_FRED_DATA = True` in the first cell when the caches should be updated from their data services.


In [16]:
import importlib

import manufacturing_stress_forecasting.data as manufacturing_stress_data
import manufacturing_stress_forecasting.features as manufacturing_stress_features

importlib.reload(manufacturing_stress_features)
manufacturing_stress_data = importlib.reload(manufacturing_stress_data)

service = manufacturing_stress_data.build_manufacturing_stress_service(
    cache_dir=REPO_ROOT / "data" / "fred",
    yahoo_cache_dir=REPO_ROOT / "data" / "yfinance",
    refresh=REFRESH_FRED_DATA,
)
print(service.summary().to_string(index=False))


                        series_id                                                                                       description                                   source                                units frequency  n_obs      start        end
                         CPIAUCSL                                                      Consumer Price Index for All Urban Consumers                          FRED (CPIAUCSL)                                Index        MS    955 1947-01-01 2026-08-01
                          CPI_YOY                                                        Consumer Price Index year-over-year change             Derived from FRED (CPIAUCSL)                              Percent        MS    943 1948-01-01 2026-08-01
                         FEDFUNDS                                                                      Effective federal funds rate                          FRED (FEDFUNDS)                              Percent        MS    866 1954-07-01 2026-08-01
    

## Load the common backtest specification

Every predictor below receives this same target, horizon, forecast-origin schedule, warmup, and cutoff-aware data service.

In [12]:
with SPEC_PATH.open() as file:
    spec = BacktestSpec.model_validate(yaml.safe_load(file))
spec = spec.model_copy(update={"stride": BACKTEST_STRIDE})

print(spec.task.description)
print(f"Origins: {spec.start.date()} to {spec.end.date()}, stride={spec.stride}, warmup={spec.warmup}")
print(f"Target: {spec.task.target_series_id}; horizon={spec.task.horizons[0]} {spec.task.frequency}")

Probability that U.S. manufacturing will be under stress three months ahead. A resolved month is stressed when IPMAN has declined by at least 2 percent over its preceding three months.
Origins: 2018-01-01 to 2024-12-01, stride=3, warmup=60
Target: manufacturing_stress; horizon=3 MS


## Run one smoke test

This is the single execution path. Before scoring, it verifies that the logistic and XGBoost predictors have all 15 registered model features: the three IPMAN momentum features plus the twelve FRED/Yahoo macro features, including 3-month and 12-month monthly-origin returns for both SPY and XLI. All are monthly model frames; daily and weekly source series are reduced to monthly observations first. The cell then runs the historical-frequency, logistic-regression, and XGBoost predictors. Set `RUN_LLMP_BACKTEST = True` in the first cell to include the token-limited LLMP in the same run. Compatible complete results are loaded from cache unless `FORCE_REFRESH_LLMP_CACHE = True`; refreshed data also forces a rerun.


In [17]:
from manufacturing_stress_forecasting.features import FEATURE_SERIES_IDS

feature_summary = service.summary().set_index("series_id")
missing_features = sorted(set(FEATURE_SERIES_IDS) - set(feature_summary.index))
if missing_features:
    raise RuntimeError(f"Missing model features: {missing_features}")

non_monthly_features = [
    series_id for series_id in FEATURE_SERIES_IDS if feature_summary.loc[series_id, "frequency"] != "MS"
]
if non_monthly_features:
    raise RuntimeError(f"Model features are not monthly: {non_monthly_features}")

print(
    "Logistic/XGBoost feature contract verified: "
    f"{len(FEATURE_SERIES_IDS)} features ({len(FEATURE_SERIES_IDS) - 3} FRED/Yahoo macro features) "
    "are registered at monthly frequency."
)
print("Daily/weekly source data, including Yahoo returns, is reduced to monthly observations before modeling.")

predictors = [
    HistoricalFrequencyPredictor(),
    ManufacturingStressLogisticPredictor(),
    ManufacturingStressXGBoostPredictor(),
]
if RUN_LLMP_BACKTEST:
    predictors.append(build_manufacturing_stress_agent_predictor(anonymize_dates=True))

results = {}
for predictor in predictors:
    if RUN_LLMP_BACKTEST:
        result = run_or_load_backtest(
            predictor,
            spec,
            service,
            force_refresh=FORCE_REFRESH_LLMP_CACHE or REFRESH_FRED_DATA,
            store_dir=STORE_DIR,
        )
    else:
        result = backtest(predictor=predictor, spec=spec, data_service=service)
    results[predictor.predictor_id] = result
    print(
        f"{result.predictor_id}: {result.mean_score:.4f} mean {result.metric}; "
        f"scored={len(result.scores)} skipped={result.skipped_origins}"
    )

validate_comparable_results(list(results.values()))
print("All predictors were scored on identical forecast origins.")


Logistic/XGBoost feature contract verified: 15 features (12 FRED/Yahoo macro features) are registered at monthly frequency.
Daily/weekly source data, including Yahoo returns, is reduced to monthly observations before modeling.
historical_frequency: 0.0356 mean brier; scored=28 skipped=0
manufacturing_stress_logistic_ipman_rates: 0.0458 mean brier; scored=28 skipped=0
manufacturing_stress_xgboost_ipman_rates: 0.0416 mean brier; scored=28 skipped=0
All predictors were scored on identical forecast origins.


## Compare results

Lower Brier score is better. The notebook refuses to compare results unless every predictor was scored on the exact same forecast-origin and forecast-date pairs.


In [14]:
comparison = pd.DataFrame(
    [
        {
            "predictor": result.predictor_id,
            "metric": result.metric,
            "mean_brier": result.mean_score,
            "scored": len(result.scores),
            "skipped": result.skipped_origins,
        }
        for result in results.values()
    ]
)
comparison.sort_values("mean_brier") if not comparison.empty else comparison

,predictor,metric,mean_brier,scored,skipped
0,historical_frequency,brier,0.035598,28,0
2,manufacturing_stress_xgboost_ipman_rates,brier,0.041616,28,0
1,manufacturing_stress_logistic_ipman_rates,brier,0.045765,28,0


## Optional current forecast

Set `RUN_CURRENT_FORECAST = True` in the first cell to run the selected predictors against the latest released IPMAN data. This is a current forecast, not a historical backtest, so it does not produce a Brier score. `RUN_LLMP_BACKTEST` controls whether the LLMP is included here too.

In [ ]:
if RUN_CURRENT_FORECAST:
    with SPEC_PATH.open() as file:
        current_task = BacktestSpec.model_validate(yaml.safe_load(file)).task

    full_ipman = service.get_series(
        IPMAN_SERIES_ID,
        as_of=pd.Timestamp("2100-01-01").to_pydatetime(),
    )
    current_as_of = pd.Timestamp(full_ipman["released_at"].max())
    current_context = service.context(as_of=current_as_of.to_pydatetime())

    current_predictors = [
        HistoricalFrequencyPredictor(),
        ManufacturingStressLogisticPredictor(),
        ManufacturingStressXGBoostPredictor(),
    ]
    if RUN_LLMP_BACKTEST:
        current_predictors.append(build_manufacturing_stress_agent_predictor())

    print(f"Forecast origin: {current_as_of.date()}")
    print(
        "Latest visible IPMAN reference month: "
        f"{pd.Timestamp(current_context.get_series(IPMAN_SERIES_ID)['timestamp'].max()).date()}"
    )
    for predictor in current_predictors:
        prediction = predictor.predict(current_task, current_context)[0]
        print({
            "predictor": prediction.predictor_id,
            "forecast_date": str(pd.Timestamp(prediction.forecast_date).date()),
            "stress_probability": prediction.payload.probability,
            "metadata": prediction.metadata,
        })
else:
    print("Current forecast is disabled. Set RUN_CURRENT_FORECAST = True to enable it.")